# 多模态大模型基础与数字员工架构设计

---

## 课程导航

| 部分 | 内容 | 对应文件 |
|------|------|----------|
| **Part 1** | 本节定位 · 从需求到可启动骨架 | — |
| **Part 2** | 业务与多模态大模型基础（导览） | — |
| **Part 3** | 架构设计：五层 + 存储 + 数据流 | — |
| **Part 4** | 配置治理与启动自检 | `core/config.py` |
| **Part 5** | 启动编排与骨架 | `core/main.py`、`core/infra/*` |
| **Part 6** | 把项目跑起来 | 终端命令 + 探活 |
| **Part 7** | 随堂练习（含答案） + 小结 | 多模块联合 |

---

> **面向学员**：本教程面向基础学员，每个知识点都有详细讲解和代码注释，
> 建议按顺序逐 Cell 运行，观察每一次的输出结果。
>
> 本节课是项目的**启动课**，不背 API，而是回答一个问题：
> **一个多模态数字员工平台，骨架该怎么搭、配置该怎么管、启动为什么必须"早失败"？**
>
> 说明：PPT 已把业务与架构讲透，本 notebook 只做**导览**，重心放在
> **`config.py` / `main.py` 代码走读 + 4 个可运行实验**。全部实验**不需要真实 API Key**。

---
## Part 1：本节定位 —— 从需求到可启动骨架
---

### 1.1 六节课全景：本节在哪里

| 节次 | 主题 | 核心产出 |
|---|---|---|
| **第 34 节** | **多模态大模型基础与数字员工架构设计** | **架构设计 + 可启动骨架（本节）** |
| 第 35 节 | 多模态核心能力开发与代码实现 | ASR / TTS / OCR / VLM 工具层 |
| 第 36 节 | 多模态与 RAG / 智能体深度融合开发 | 文本优先的多模态 RAG |
| 第 37 节 | 多模态与多智能体协同开发实现 | CrewAI 三角色 + 人机协同 |
| 第 38 节 | 多模态交互系统全流程开发与联调优化 | 三端产品 + 实时链路 + 接口层 |
| 第 39 节 | 多模态全栈评测、企业级部署与商业化交付 | 评测 + 容器化部署 + 交付文档 |

本节是**项目启动课**：先立"业务 + 架构"，再谈编码。
后面五节做的每一个模块，都挂在本节画出的这张"地图"上。

### 1.2 本节三条主线

1. **业务 → 架构**：三类输入（文字 / 语音 / 截图）× 两类出口（AI 自助解决 / 转人工或建单），
   如何落成 `api / infra / ai / rag / service` 五层结构；
2. **配置 → 治理**：用统一「能力四元组」把模型能力全部外置到 `.env`，换供应商只改配置、代码零改动；
3. **入口 → 启动**：`main.py` 的 lifespan 启动编排，以及"必填项缺失就拒绝启动"的 fail-fast 自检。

### 1.3 两条贯穿全节的统一约定

1. **本项目聚焦"理解 + 表达"类多模态**：ASR 听懂、OCR 读字、VLM 看懂、TTS 说出；
   **不做文生图**——IT 服务台要的是"看懂用户给的图"，不是"画一张图给用户"。
2. **所有模型能力都走 OpenAI 兼容协议**：嵌入 / 精排 / 视觉理解 / 语音识别一律用
   `{能力}_BASE_URL + {能力}_MODEL + {能力}_API_KEY` 描述，换供应商只改变量。

---
## 环境准备（本节通用）
---

> 本节所有代码单元都依赖下面这一次环境检查：确认 Python 与依赖就绪、
> 安装 Jupyter 内的 `asyncio` 兼容补丁，并定义 `read()` / `show_around()` 两个读源码小工具，
> 以及 Part 4 要用到的**配置脱敏打印工具**。
>
> **本节实验不需要真实 API Key**：配置实验用"抹掉指定项"的方式模拟缺配置（不动你的 `.env`），
> 启动实验只探 `/api/health`。嵌入维度自检失败也只会**告警**、不会阻塞启动。

In [1]:
# ============================================================
# Cell 1：环境检查 + asyncio 兼容补丁 + 读源码工具
# ============================================================
import sys
import importlib

print("=" * 60)
print("  环境检查")
print("=" * 60)
print(f"  Python 版本: {sys.version.split()[0]}")
print(f"  解释器路径: {sys.executable}")
print()

for pkg, imp in {
    "fastapi": "fastapi",                      # 后端框架（应用装配 / 路由）
    "uvicorn": "uvicorn",                      # ASGI 服务器（Part 6 用它起服务）
    "pydantic": "pydantic",                    # 配置模型
    "pydantic-settings": "pydantic_settings",  # .env 加载
}.items():
    try:
        importlib.import_module(imp)
        print(f"     [OK] {pkg}")
    except ImportError:
        print(f"     [NO] {pkg} 未安装")

from pathlib import Path
print()
print(f"  .env 存在：        {Path('.env').exists()}")
print(f"  .env.example 存在：{Path('.env.example').exists()}")
print(f"  core/ 存在：       {Path('core').exists()}")

# 把项目根目录加入 import 路径，保证能 import core.*
sys.path.insert(0, ".")
print(f"  已把当前目录加入 import 路径：{sys.path[0]}")


# ------------------------------------------------------------------
# 兼容补丁：让 asyncio.run() 在"已运行事件循环"的内核里也能用（幂等·可重入）
# ------------------------------------------------------------------
import asyncio
import concurrent.futures


def _install_asyncio_run_compat() -> bool:
    """安装兼容补丁；已安装则返回 False（不重复打补丁）。"""
    if getattr(asyncio, "_mde_run_async_patched", False):
        return False

    asyncio._mde_orig_run = asyncio.run   # 原始实现挂在模块属性上，后续不会被覆盖

    def run_async(coro, **kwargs):
        """在 notebook 中安全执行协程（兼容"已运行事件循环"的内核）。"""
        orig = asyncio._mde_orig_run      # 每次取模块上的原始实现，确保不递归
        try:
            asyncio.get_running_loop()
        except RuntimeError:
            return orig(coro, **kwargs)   # 常规路径：无运行中的循环，直接跑
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: orig(coro, **kwargs)).result()

    asyncio.run = run_async
    asyncio._mde_run_async_patched = True
    return True


if _install_asyncio_run_compat():
    print("\n已启用 run_async 兼容：asyncio.run() 现在可用于运行中的事件循环内核。")
else:
    print("\nrun_async 兼容已启用（跳过重复补丁）。")


# ------------------------------------------------------------------
# 读源码小工具：全节通用（read / find / show / show_around）
# ------------------------------------------------------------------
# 后续所有"源码走读"单元都用这几个函数，避免把整份文件贴进 notebook：
#   read(path)                读全文
#   find(path, needle)        找锚点所在行号（1-based）
#   show(path, s, e, label)   打印 [s, e] 行区间
#   show_around(path, anchor, before, after, label)   以锚点为中心打印
def read(path):
    """读文本文件内容（UTF-8）。"""
    return Path(path).read_text(encoding="utf-8")


def find(path, needle, nth=1):
    """返回 needle 第 nth 次出现的行号（1-based）；找不到抛 ValueError。

    用"锚点 + 偏移"而不是写死行号，是为了让源码改动后本 notebook 依然能对上。
    """
    hits = [i + 1 for i, line in enumerate(read(path).splitlines()) if needle in line]
    if len(hits) < nth:
        raise ValueError(f"{path} 中找不到第 {nth} 次出现的 {needle!r}")
    return hits[nth - 1]


def show(path, start=1, end=None, label=""):
    """打印文件 [start, end] 行（1-based，含端点）——用于展示关键源码片段。"""
    lines = read(path).splitlines()
    start = max(1, start)
    end = min(end or len(lines), len(lines))
    print("=" * 68)
    print(f"  {path}" + (f"  —— {label}" if label else "")
          + f"  （第 {start}~{end} 行 / 共 {len(lines)} 行）")
    print("=" * 68)
    for i in range(start, end + 1):
        print(f"{i:>4}| {lines[i-1]}")
    print()


def show_around(path, anchor, before=0, after=18, label=""):
    """以 anchor 所在行为基准，打印 [anchor-before, anchor+after] 行。"""
    line = find(path, anchor)
    show(path, line - before, line + after, label=label or f"锚点 {anchor!r}")

print("已定义：read() / find() / show() / show_around()")

  环境检查
  Python 版本: 3.13.7
  解释器路径: c:\Users\13691\Documents\Code\260601-GPKJ\07-MLM\.venv\Scripts\python.exe

     [OK] fastapi
     [OK] uvicorn
     [OK] pydantic
     [OK] pydantic-settings

  .env 存在：        True
  .env.example 存在：True
  core/ 存在：       True
  已把当前目录加入 import 路径：.

已启用 run_async 兼容：asyncio.run() 现在可用于运行中的事件循环内核。
已定义：read() / find() / show() / show_around()


In [2]:
# ============================================================
# Cell 2：配置脱敏打印工具（Part 4 实验①用）
# ============================================================
from pathlib import Path

from core.config import settings, Settings

SECRET_HINTS = ("API_KEY", "SECRET", "PASSWORD", "TOKEN")


def is_secret(name: str) -> bool:
    """字段名里带 API_KEY / SECRET / PASSWORD / TOKEN 的，一律视为敏感字段。"""
    upper = name.upper()
    return any(hint in upper for hint in SECRET_HINTS)


def mask(value) -> str:
    """密钥一律完全遮住，只保留长度信息——课堂上可以放心打印。"""
    if value in (None, ""):
        return "(空)"
    return f"{'*' * 8}（已隐藏，共 {len(str(value))} 字符）"


def dump_settings() -> None:
    """按声明顺序打印当前生效的配置（敏感字段自动打码）。"""
    env_file = Path(str(Settings.model_config.get("env_file") or ""))
    try:
        shown = env_file.relative_to(Path.cwd())     # 在仓库根运行时显示为 .env
    except ValueError:
        shown = env_file
    print("=" * 66)
    print(f"  当前生效配置（来源：{shown}）")
    print("=" * 66)
    for name in Settings.model_fields:
        value = getattr(settings, name)
        text = mask(value) if is_secret(name) else (value if value != "" else "(空)")
        print(f"  {name:<28} = {text}")
    print()


print("已定义：is_secret() / mask() / dump_settings()")
print(f"  自检一下：is_secret('VLM_API_KEY') = {is_secret('VLM_API_KEY')}，"
      f"is_secret('LLM_MODEL') = {is_secret('LLM_MODEL')}")

已定义：is_secret() / mask() / dump_settings()
  自检一下：is_secret('VLM_API_KEY') = True，is_secret('LLM_MODEL') = False


---
## Part 2：业务场景与多模态大模型基础
---

> 这一 Part 是**导览**：PPT 已展开讲过，这里只留下"写代码时要用到的结论"。

### 2.1 业务场景：三类输入、两类出口

```
          ┌── 文字 ──┐
 员工提问 ─┼── 语音 ──┼──►  AI 数字员工  ──┬──► ① 自助解决（带图回复 + 参考来源）
          └── 截图 ──┘                     ├──► ② 转人工坐席
                                          └──► ③ 自动建工单（②③ 可并存）
```

| 项 | 内容 |
|---|---|
| 输入 | 文字 / 语音 / 截图（覆盖绝大多数员工的提问方式） |
| 出口 | AI 自助解决、转人工坐席、自动建工单 |
| 典型场景 | VPN 连不上、密码重置、软件安装申请、报错截图诊断 |
| 痛点 | 响应不及时、口径不统一、重复工单多 |
| 技术抓手 | 多模态大模型 + RAG + 智能体 + 人机协同 |

### 2.2 交付标准：三个闭环

| 闭环 | 验收口径 |
|---|---|
| 用户闭环 | 文字 / 语音 / 截图 → **带图回复 + 参考来源** |
| 协同闭环 | **AI ↔ 人工双向转接**（转人工 → 坐席接管 → 转回 AI） |
| 工程闭环 | 可本地 / Docker 部署、一键评测、`pytest` 全绿回归 |

> 记住这三个闭环——后面每节课的"交付物 / 验收"都是它们的分解。

### 2.3 多模态能力全景：理解类 vs 生成类

| 能力 | 方向 | 解决什么问题 | 本项目用谁 |
|---|---|---|---|
| ASR | 语音 → 文本 | 把"说不清"的口述变成可检索的文字 | 在线 `SenseVoice` / 本地 `funasr` |
| OCR | 图片 → 文字 | 把截图里的字抠出来 | 本地 `PaddleOCR` / 复用 VLM |
| VLM | 图片 → 结构化结论 | 看懂报错截图：错误码、错误类型 | `Qwen3-VL-30B-A3B-Instruct` |
| TTS | 文本 → 语音 | 把回复念出来（语音播报） | `edge-tts`（本地、免费） |
| 嵌入 | 文本 → 向量 | 让知识"可被语义检索" | `Qwen3-VL-Embedding-8B` |
| 精排 | 查询 + 候选 → 分数 | 从召回结果里挑最相关的几条 | `Qwen3-VL-Reranker-8B` |

> **边界很重要**：本项目**不做文生图**。服务台要的是"看懂用户给的图"，
> 不是"画一张图给用户"，所以只需"理解类"（ASR / OCR / VLM）与"表达类"（TTS）。

### 2.4 技术选型的设计考量

| 考量 | 本项目选择 | 为什么 |
|---|---|---|
| 可本地 · 可私有化 | TTS 用 `edge-tts`、向量库用 ChromaDB、关系库用 SQLite | 零容器依赖，教学与私有化部署都省事 |
| 换供应商不改代码 | 全部走 **OpenAI 兼容协议** +「能力四元组」 | 只改 `.env`，代码一行不动 |
| 统一多模态语义空间 | 嵌入统一用 `Qwen3-VL-Embedding-8B` | 文本 / 表格 / 图片描述进**同一个**向量空间，避免跨模态不可比 |
| 密钥集中治理 | 全部集中 `.env`，仓库只留 `.env.example` | 敏感信息不进代码仓库 |

> 这四条"考量"会在 Part 4 变成 `config.py` 里的**实际代码**。

---
## Part 3：架构设计 —— 五层 + 存储 + 数据流
---

### 3.1 五层架构总览

```
                        ┌────────────────────────────────┐
                        │  入口 core/main.py              │
                        │  lifespan 启动编排（Part 5）     │
                        └────────────────┬───────────────┘
        ┌──────────────┬────────────────┼────────────────┬──────────────┐
        ▼              ▼                ▼                ▼              ▼
 ┌────────────┐ ┌────────────┐ ┌────────────┐ ┌────────────┐ ┌────────────┐
 │  路由层     │ │ 基础设施层  │ │ 智能编排层  │ │多模态RAG层  │ │ 业务编排层  │
 │ core/api/  │ │core/infra/ │ │  core/ai/  │ │  core/rag/ │ │core/service│
 ├────────────┤ ├────────────┤ ├────────────┤ ├────────────┤ ├────────────┤
 │ 五组接口：  │ │ 库·鉴权·    │ │ 统一模型    │ │ 解析·嵌入·  │ │ 会话流转·   │
 │ auth/chat/ │ │ 中间件·媒体·│ │ 客户端·     │ │ 向量库·     │ │ 转人工·     │
 │ agent/     │ │ 实时通信·   │ │ 三角色      │ │ 检索编排    │ │ 统计·       │
 │ admin/ws   │ │ 定时·MCP    │ │ Agent       │ │            │ │ 种子数据    │
 └────────────┘ └────────────┘ └────────────┘ └────────────┘ └────────────┘
        │                                                            │
        └──────────────►  前端三端 SPA ／ 工单 MCP Server  ◄──────────┘
```

### 3.2 各层职责与边界

| 层 | 目录 | 只做分内事 | 不做什么 |
|---|---|---|---|
| 路由层 | `core/api/` | 收请求、鉴权、调服务、返回 | 不写业务逻辑、不直接写 SQL |
| 基础设施层 | `core/infra/` | 库 / 鉴权 / 媒体 / 实时 / 调度 / MCP 桥接 | 不感知业务语义 |
| 智能编排层 | `core/ai/` | 模型调用、多角色 Agent、工具装配 | 不碰数据库与 HTTP |
| 多模态 RAG 层 | `core/rag/` | 解析、嵌入、向量库、检索编排 | 不管会话状态 |
| 业务编排层 | `core/service/` | 会话流转、转人工判定、统计 | 不直接调模型 SDK |

> **分层的教学意义**：每一层都能**单独讲、单独换**。
> 比如把 ChromaDB 换成别的向量库，只需动 `core/rag/vector_store.py`，上层无感。

### 3.3 源码目录 = 教学模块 = 课次（全项目地图）

| 目录 / 文件 | 层 | 主责课次 |
|---|---|---|
| `core/main.py`、`core/config.py` | 入口 | **第 34 节（本节）** |
| `core/infra/db.py`、`init_data.py`、`scheduler.py` | 基础设施层 | **第 34 节（本节）** |
| `core/infra/{auth,middleware}.py` | 基础设施层 | 第 38 节（本节只点名） |
| `core/infra/media.py` | 基础设施层 | 第 35 节 |
| `core/infra/{mcp_client,mcp_bridge,async_bridge}.py` | 基础设施层 | 第 37 节 |
| `core/infra/ws_manager.py` | 基础设施层 | 第 38 节 |
| `core/ai/llm.py` | 智能编排层 | 第 35 节 |
| `core/ai/{crew_agents,crew_tools}.py` | 智能编排层 | 第 37 节 |
| `core/rag/multimodal.py` | 多模态 RAG 层 | 第 35 节 |
| `core/rag/{parsers,embedding,vector_store,multimodal_rag}.py` | 多模态 RAG 层 | 第 36 节 |
| `core/service/seed_knowledge.py` | 业务编排层 | 第 36 节 |
| `core/service/{handoff,session_flow,session_router,turn_context}.py` | 业务编排层 | 第 37 节 |
| `core/service/analytics.py` | 业务编排层 | 第 38 节 |
| `core/api/*.py` | 路由层 | 第 38 节（本节只点名） |
| `mcp_servers/it_service/server.py` | 外部系统 | 第 37 节 |
| `frontend/src/` | 前端三端 | 第 38 节 |
| `eval/`、`deploy/`、`tests/` | 评测 · 部署 · 测试 | 第 39 节 |

> Part 5 的**实验③**会用 `ast` 把这张表**自动扫出来**，亲眼确认目录结构。

### 3.4 三级存储架构

| 存储 | 技术 | 放什么 | 关键约束 |
|---|---|---|---|
| 向量库 | ChromaDB（嵌入式） | 知识库分块的向量 | **单进程独占**：服务必须单实例，不能开多 worker |
| 关系库 | SQLite | `app.db` 业务数据、`tickets.db` 工单数据 | 两个库**分开**，工单库归 MCP Server 管 |
| 媒体目录 | 本地文件 | 图片 / 音频**原文件** | 用 `asset_id` 寻址，经**带鉴权**的 URL 访问 |

> 为什么向量库要"单实例"？嵌入式 ChromaDB 是对本地文件的独占访问，
> 多个进程同时写会互相破坏——这条约束直接决定了后面的部署形态（第 39 节）。

### 3.5 数据模型总览（讲关系，不讲字段）

```
users ──1:N──► conversations ──1:N──► messages
  │                  │
  │                  └──1:N──► tool_calls        （每次工具调用留痕，形成审计链）
  └──1:1──► agent_profiles                      （坐席档案：在线状态 / 接待上限）

knowledge_documents                              （知识库文档元数据；向量在 ChromaDB）
tickets（tickets.db，独立库）                      （经 conversation_id / user_id 关联，不做级联）
```

| 关系 | 说明 |
|---|---|
| 用户 → 会话 | 一个员工可有多个会话；会话带 `title`、`last_message_at` 供列表排序 |
| 会话 → 消息 | 消息带 `modality`（text/audio/image）与 `intent`，多模态与意图都落在消息上 |
| 会话 → 工具调用 | 每次检索 / 建单都落一条，是**审计与可观测**的依据 |
| 坐席 → 档案 | 在线状态与接待数决定"能不能被派单" |
| 工单 ↔ 会话 | **独立生命周期**，只关联不级联（会话关了工单不一定关） |

### 3.6 端到端数据流（本节回扣）

```
   ①感知            ②识别            ③检索           ④编排          ⑤交付
 ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────────┐
 │ 语音/截图 │──►│ 意图+情绪 │──►│ 知识库检索│──►│ 多 Agent │──►│ 实时推送前端   │
 │ ASR/VLM  │   │perceive_*│   │   RAG    │   │  CrewAI  │   │  WS / REST   │
 └──────────┘   └──────────┘   └──────────┘   └────┬─────┘   └──────────────┘
                                                   │
                                             ┌─────▼─────┐
                                             │ 工单 MCP   │
                                             │ 转人工协同  │
                                             └───────────┘
```

| 环节 | 归属 | 课次 |
|---|---|---|
| ① 感知（ASR / OCR / VLM） | `core/rag/multimodal.py`、`core/infra/media.py` | 第 35 节 |
| ② 意图 + 情绪识别 | `perceive_message()` | 第 37 节 |
| ③ 知识库检索 | `core/rag/multimodal_rag.py` | 第 36 节 |
| ④ 多 Agent 编排 | `core/ai/crew_agents.py` | 第 37 节 |
| ⑤ 交付（WS / REST / 工单 / 转人工） | `core/api/*`、`core/service/*` | 第 37 / 38 节 |

### 3.7 一个外部系统怎么接：MCP 的定位

本项目的**唯一外部系统是工单系统**：它是一个独立的 stdio 进程，
通过 **MCP（Model Context Protocol）** 对外暴露 8 个工具
（建单 / 查单 / 改状态 / 重置密码 / 申请软件权限 …）。

```
 中台进程 core/                     独立进程 mcp_servers/it_service/server.py
 ┌──────────────────┐   stdio/MCP   ┌──────────────────────────┐
 │ mcp_client.py    │◄─────────────►│ 8 个工单工具              │
 │ mcp_bridge.py    │   常驻连接     │ tickets.db（独立 SQLite） │
 └──────────────────┘               └──────────────────────────┘
```

| 设计点 | 说明 |
|---|---|
| 为什么用独立进程 | 工单系统天然是"外部系统"，独立进程 = 真实解耦，不是同进程的假模块 |
| 为什么用常驻连接 | MCP 客户端有生命周期约束（建连/关闭都有成本），每来一个请求就重连不可行 |
| 桥接问题 | Agent 工具是**同步**调用，而给前端推送是**异步**的 → 需要 `async_bridge` 跨线程桥接 |

> 所以在 Part 5 你会看到 `main.py` 启动时**专门有一行 `set_main_loop(...)`**：
> 目的就是让同步的工具回调能找回主事件循环。这是"架构决定代码"的典型例子。

---
## Part 4：配置治理与启动自检（`core/config.py` · A 档）
---

### 4.1 「能力四元组」思想

本项目要用到 6 种模型能力（LLM / 嵌入 / 精排 / 视觉理解 / ASR / OCR）。
如果每种能力都各起一套变量名，代码里就会散落各种读配置的分支。

本项目统一成**四元组**：

```
{能力}_PROVIDER    走哪家 / 哪种协议（默认 openai_compatible）
{能力}_BASE_URL    接入地址（换供应商只改这里）
{能力}_MODEL       模型名
{能力}_API_KEY     密钥（留空则继承 SILICONFLOW_API_KEY）
```

于是"换供应商"从**改代码**变成了**改环境变量**：

```bash
# 把视觉理解从硅基流动换成别家，只改这 3 行
VLM_BASE_URL=https://api.example.com/v1
VLM_MODEL=some-vl-model
VLM_API_KEY=sk-xxx
```

### 4.2 先看模块的"说明书"

读代码前先读 `docstring`——它是模块的对外说明书。先有地图，再看细节。

In [3]:
# ============================================================
# Cell 3：config.py 的模块说明书 + 顶层定义清单（先有地图）
# ============================================================
import ast

path = "core/config.py"
tree = ast.parse(read(path))

print("=" * 66)
print("  core/config.py —— 模块说明书")
print("=" * 66)
print(ast.get_docstring(tree))

print("=" * 66)
print("  顶层定义（按源码顺序）")
print("=" * 66)
for node in tree.body:
    if isinstance(node, ast.ClassDef):
        print(f"  class   {node.name:<24} (第 {node.lineno} 行)")
    elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        print(f"  def     {node.name:<24} (第 {node.lineno} 行)")
    elif isinstance(node, ast.Assign):
        for tgt in node.targets:
            if isinstance(tgt, ast.Name) and (tgt.id.isupper() or tgt.id == "ENV_FILE"):
                print(f"  常量     {tgt.id:<24} (第 {node.lineno} 行)")

print()
print(f"  文件总行数：{len(read(path).splitlines())}")
print("  规律：Settings 类承载「配置字段」，_REQUIRED / ConfigError / validate_config 承载「启动自检」。")

  core/config.py —— 模块说明书
统一配置加载（能力四元组），见设计文档第十七章。

命名规范：{能力}_PROVIDER / {能力}_BASE_URL / {能力}_MODEL / {能力}_API_KEY
全部走 OpenAI 兼容协议，替换供应商只改变量、不改代码。
  顶层定义（按源码顺序）
  常量     PROJECT_ROOT             (第 18 行)
  常量     ENV_FILE                 (第 19 行)
  class   Settings                 (第 22 行)
  def     get_settings             (第 176 行)
  class   ConfigError              (第 194 行)
  def     validate_config          (第 198 行)
  def     _check_embedding_dim     (第 219 行)

  文件总行数：238
  规律：Settings 类承载「配置字段」，_REQUIRED / ConfigError / validate_config 承载「启动自检」。


### 4.3 走读四个关键片段

`config.py` 不长，但每一段都对应一个设计决策。下面四段是重点。

In [4]:
# ============================================================
# Cell 4：config.py 四个关键片段
# ============================================================
show_around("core/config.py", "# .env 固定用", before=0, after=14,
            label="① 为什么用「仓库根绝对路径」加载 .env")

show_around("core/config.py", "def _resolve", before=7, after=24,
            label="② Key 回退链：能力 Key 留空 → 继承 SILICONFLOW_API_KEY")

show_around("core/config.py", "_REQUIRED: list", before=2, after=27,
            label="③ 启动自检：必填清单 + ConfigError（fail-fast）")

show_around("core/config.py", "def _check_embedding_dim", before=0, after=22,
            label="④ 嵌入维度自检：模型返回维度必须等于 EMBEDDING_DIM")

  core/config.py  —— ① 为什么用「仓库根绝对路径」加载 .env  （第 13~27 行 / 共 238 行）
  13| # .env 固定用「仓库根」的绝对路径加载。
  14| # 若写成相对路径 ".env"，会随进程工作目录（CWD）变化：在非仓库根目录启动的
  15| # Jupyter 内核 / 服务可能读不到本项目 .env（配置静默退回默认值），甚至误读其它
  16| # 目录下的同名文件（例如带行内注释的旧副本 → 解析报 ValidationError）。
  17| # 用绝对路径可让「读的是哪个 .env」不再依赖 CWD。
  18| PROJECT_ROOT = Path(__file__).resolve().parents[1]
  19| ENV_FILE = PROJECT_ROOT / ".env"
  20| 
  21| 
  22| class Settings(BaseSettings):
  23|     model_config = SettingsConfigDict(
  24|         env_file=str(ENV_FILE),
  25|         env_file_encoding="utf-8",
  26|         extra="ignore",
  27|         case_sensitive=True,

  core/config.py  —— ② Key 回退链：能力 Key 留空 → 继承 SILICONFLOW_API_KEY  （第 138~169 行 / 共 238 行）
 138|     MCP_SERVER_SCRIPT: str = "mcp_servers/it_service/server.py"
 139|     TICKETS_DB_PATH: str = ""
 140| 
 141|     # ------------------------------------------------------------------
 142|     # Key 回退：{能力}_API_KEY 留空时继承 SILICONFLOW_API_KEY
 143|     # 优先级：{能力}_API_KEY > 

### 4.4 实验①：把 `.env` 读成了什么？（脱敏打印）

配置最常见的问题是"我明明改了 `.env`，怎么不生效？"。
最有效的排查方式是：**把内存里真正生效的 `settings` 打出来看一眼**。

下面这个小工具会把密钥**打码**后再打印，可以放心在课堂上运行。

In [5]:
# ============================================================
# Cell 5：实验① —— 脱敏打印当前生效的配置 + 验证 Key 回退链
# ============================================================
dump_settings()

print("=" * 66)
print("  Key 回退链验证（只看「有没有」与打码结果，不打印明文）")
print("=" * 66)


def key_source(cap: str) -> str:
    """判断该能力的生效 Key 来自哪一级回退。"""
    if getattr(settings, f"{cap}_API_KEY", ""):
        return f"{cap}_API_KEY"
    if settings.SILICONFLOW_API_KEY:
        return "SILICONFLOW_API_KEY"
    if cap == "VLM":
        return "LLM_API_KEY（VLM 的第三级回退）"
    return "(未配置)"


for cap in ("LLM", "EMBEDDING", "RERANKER", "VLM", "ASR"):
    print(f"  {cap:<10} 生效 Key = {mask(getattr(settings, cap.lower() + '_key')):<20}"
          f" 来源 = {key_source(cap)}")

print()
print("  结论：业务代码只读 settings.llm_key / embedding_key / ...，")
print("        完全不用关心密钥到底写在哪个变量里——回退链在配置层就收口了。")

  当前生效配置（来源：.env）
  LLM_PROVIDER                 = openai_compatible
  LLM_BASE_URL                 = https://api.siliconflow.cn/v1
  LLM_MODEL                    = deepseek-ai/DeepSeek-V4-Flash
  LLM_API_KEY                  = (空)
  EMBEDDING_PROVIDER           = openai_compatible
  EMBEDDING_BASE_URL           = https://api.siliconflow.cn/v1
  EMBEDDING_MODEL              = Qwen/Qwen3-VL-Embedding-8B
  EMBEDDING_API_KEY            = (空)
  EMBEDDING_DIM                = 1024
  EMBEDDING_MAX_IMAGE_SIDE     = 1024
  EMBEDDING_MAX_IMAGE_BYTES    = 32000
  RERANKER_ENABLED             = True
  RERANKER_BASE_URL            = https://api.siliconflow.cn/v1
  RERANKER_MODEL               = Qwen/Qwen3-VL-Reranker-8B
  RERANKER_API_KEY             = (空)
  RERANKER_TOP_N               = 3
  VLM_BASE_URL                 = https://api.siliconflow.cn/v1
  VLM_MODEL                    = Qwen/Qwen3-VL-30B-A3B-Instruct
  VLM_API_KEY                  = (空)
  ASR_PROVIDER                 = api
  ASR_BAS

### 4.5 实验②：让启动自检"故意失败"一次

`validate_config()` 是**启动自检**：必填项缺失就抛 `ConfigError`，
而且必须能**定位到"哪个能力、哪个变量"**。这在真实排障时非常关键——
报错不能只说一句"配置错误"。

下面用一份"**抹掉指定项的临时配置**"来触发它，**完全不动你的 `.env`**。

In [6]:
# ============================================================
# Cell 6：实验② —— 构造缺项配置，观察 ConfigError 的定位能力
# ============================================================
import core.config as cfg


def simulate_missing(**blanks):
    """用「当前 .env 的值 + 抹掉指定项」造一份临时配置，跑一次启动自检。

    关键点：把 _env_file 设为 None，否则 pydantic 会重新读 .env 把抹掉的项补回来。
    """
    fields = {name: getattr(settings, name) for name in cfg.Settings.model_fields}
    fields.update(blanks)
    fields["_env_file"] = None            # 不再读 .env
    probe = cfg.Settings(**fields)

    saved = cfg.settings                  # 临时替换全局单例，让 validate_config 校验这份配置
    cfg.settings = probe
    try:
        cfg.validate_config(check_embedding=False)   # 不联网，只看必填项
        return "通过（未报错）"
    except cfg.ConfigError as exc:
        return str(exc)
    finally:
        cfg.settings = saved              # 无论成败都还原，不影响后续单元


for blanks, desc in [
    ({"LLM_MODEL": ""}, "场景 1：只缺 LLM_MODEL"),
    ({"EMBEDDING_MODEL": ""}, "场景 2：只缺 EMBEDDING_MODEL"),
    ({"LLM_MODEL": "", "EMBEDDING_BASE_URL": "",
      "LLM_API_KEY": "", "SILICONFLOW_API_KEY": ""}, "场景 3：缺多项（含所有密钥）"),
]:
    print("=" * 66)
    print(f"  {desc}")
    print("=" * 66)
    print(simulate_missing(**blanks))
    print()

print("  注意报错格式：[能力] 变量名 —— 一眼就能定位到该改哪一行。")

  场景 1：只缺 LLM_MODEL
配置缺失，请检查 .env：
  - [LLM] LLM_MODEL

  场景 2：只缺 EMBEDDING_MODEL
配置缺失，请检查 .env：
  - [EMBEDDING] EMBEDDING_MODEL

  场景 3：缺多项（含所有密钥）
配置缺失，请检查 .env：
  - [LLM] LLM_MODEL
  - [EMBEDDING] EMBEDDING_BASE_URL
  - [LLM] LLM_API_KEY / SILICONFLOW_API_KEY（至少配置一个）

  注意报错格式：[能力] 变量名 —— 一眼就能定位到该改哪一行。


---
## Part 5：启动编排与骨架（`core/main.py` · A 档 + B/C 档）
---

### 5.1 lifespan：一次启动，五步编排

FastAPI 的 `lifespan` 是应用的"开机脚本"：服务开始接收请求前做准备工作，
服务退出前做资源回收。本项目把启动拆成 **5 步**，顺序不能乱：

| 步 | 做什么 | 为什么是这个顺序 |
|---|---|---|
| 0 | `set_main_loop(...)` 注册主事件循环 | 后面 Agent 的**同步工具**要回调异步广播，必须先有循环可找 |
| 1 | 配置自检（fail-fast） | **第一件事**——配置都不对，后面做什么都是白搭 |
| 2 | 建库 + 种演示账号（差量） | 有了库才能登录、才能存会话 |
| 3 | 预建工单库 + 拉起 MCP 常驻连接 | 工单是唯一外部系统，要提前握手 |
| 4 | 启动定时任务（+ 评测中断恢复） | 会话超时关闭、数据保留清理 |
| 5 | 挂载 REST / WS 路由 + 前端静态资源 | 最后才对外提供服务 |

In [7]:
# ============================================================
# Cell 7：main.py —— 启动编排（lifespan 五步）
# ============================================================
show_around("core/main.py", "@asynccontextmanager", before=2, after=15,
            label="① 步骤 0：注册主循环；步骤 1：配置自检（fail-fast，可降级为告警）")

show_around("core/main.py", "# 2. 业务库与演示账号", before=0, after=4,
            label="② 步骤 2：建库 + 差量补种演示账号")

show_around("core/main.py", "# 3. 媒体目录", before=0, after=22,
            label="③ 步骤 3：媒体目录 + 预建工单库 + MCP 常驻连接（含失败降级）")

show_around("core/main.py", "# 4. 定时任务", before=0, after=19,
            label="④ 步骤 4 + finally：启动定时任务、恢复中断评测、退出时有序回收资源")

  core/main.py  —— ① 步骤 0：注册主循环；步骤 1：配置自检（fail-fast，可降级为告警）  （第 39~56 行 / 共 172 行）
  39| 
  40| 
  41| @asynccontextmanager
  42| async def lifespan(app: FastAPI):
  43|     # 注册主事件循环，供 CrewAI 同步工具回调 async 能力
  44|     set_main_loop(asyncio.get_running_loop())
  45| 
  46|     # 1. 配置自检
  47|     try:
  48|         for warning in validate_config(check_embedding=True):
  49|             logger.warning("配置告警：%s", warning)
  50|     except ConfigError as exc:
  51|         if os.getenv("MDE_SKIP_CONFIG_CHECK") == "1":
  52|             logger.error("配置自检未通过（已按 MDE_SKIP_CONFIG_CHECK=1 跳过）：\n%s", exc)
  53|         else:
  54|             logger.critical("配置自检未通过，服务拒绝启动：\n%s", exc)
  55|             raise
  56| 

  core/main.py  —— ② 步骤 2：建库 + 差量补种演示账号  （第 57~61 行 / 共 172 行）
  57|     # 2. 业务库与演示账号（差量补种）
  58|     await db.init()
  59|     result = await init_users()
  60|     logger.info("演示账号初始化：新建 %s，已存在 %s", result["created"] or "无", result["skipped"] or "无")
  61| 

  core/main.py  —— 

### 5.2 应用装配：一个 app + 五个 router

启动编排之后，`main.py` 还负责把"整个应用"装配起来。四件事：
创建 `FastAPI` 实例、加 CORS 白名单、挂五组路由、处理前端静态资源。

In [8]:
# ============================================================
# Cell 8：main.py —— 应用装配（CORS / 五组路由 / 健康检查 / SPA fallback）
# ============================================================
show_around("core/main.py", "app = FastAPI(", before=0, after=22,
            label="① 创建 app + CORS 白名单 + include_router ×5")

show_around("core/main.py", '@app.get("/api/health")', before=0, after=11,
            label="② 健康检查：一眼看出 LLM / MCP 是否就绪（Part 6 会用到）")

show_around("core/main.py", "_dist = PROJECT_ROOT", before=0, after=17,
            label="③ SPA fallback：先判文件存在 → 再判保留前缀 → 最后回落 index.html")

  core/main.py  —— ① 创建 app + CORS 白名单 + include_router ×5  （第 105~127 行 / 共 172 行）
 105| app = FastAPI(title="MDE-LAB 多模态数字员工平台", version="1.0.0", lifespan=lifespan)
 106| 
 107| app.add_middleware(
 108|     CORSMiddleware,
 109|     allow_origins=[
 110|         "http://localhost:3000",
 111|         "http://127.0.0.1:3000",
 112|         "http://localhost:8000",
 113|         "http://127.0.0.1:8000",
 114|     ],
 115|     allow_credentials=True,
 116|     allow_methods=["*"],
 117|     allow_headers=["*"],
 118| )
 119| 
 120| # ---- 路由 ----
 121| app.include_router(auth_api.router)
 122| app.include_router(chat_api.router)
 123| app.include_router(agent_api.router)
 124| app.include_router(admin_api.router)
 125| app.include_router(ws_api.router)
 126| 
 127| 

  core/main.py  —— ② 健康检查：一眼看出 LLM / MCP 是否就绪（Part 6 会用到）  （第 128~139 行 / 共 172 行）
 128| @app.get("/api/health")
 129| async def health() -> dict:
 130|     return {
 131|         "status": "ok",
 132|         "llm_configu

### 5.3 B 档：三个撑起骨架的基础设施模块

`main.py` 调用的这几个模块，是"骨架"的实体。看懂它们，就理解了项目的启动期。

| 模块 | 一句话职责 | 值得学的设计 |
|---|---|---|
| `core/infra/db.py` | SQLite 业务库访问器（唯一的数据仓库封装层） | 同步 `sqlite3` 用 `asyncio.to_thread` 包装；写操作用进程级锁串行；**差量补列**代替迁移框架 |
| `core/infra/init_data.py` | "种"一批开箱即用的演示账号 | **差量补种**：只建不存在的，重复启动不覆盖、不报错 |
| `core/infra/scheduler.py` | 定时任务（超时关会话 / 清理媒体 / 清理会话） | `interval` 用于"要及时"的任务、`cron` 用于"半夜跑"的清理 |

In [9]:
# ============================================================
# Cell 9：B 档三模块的关键片段
# ============================================================
show_around("core/infra/db.py", "_NEW_COLUMNS", before=4, after=20,
            label="① 差量补列：CREATE TABLE IF NOT EXISTS 不会改老表，新增列必须显式补")

show_around("core/infra/init_data.py", "async def init_users", before=0, after=28,
            label="② 差量补种：账号已存在则跳过，保证反复重启不覆盖")

show_around("core/infra/scheduler.py", "def start_scheduler", before=0, after=18,
            label="③ 三类定时任务：interval（5 分钟轮询）+ cron（每日 03:00 / 03:30）")

  core/infra/db.py  —— ① 差量补列：CREATE TABLE IF NOT EXISTS 不会改老表，新增列必须显式补  （第 315~339 行 / 共 1021 行）
 315| 
 316|     # 差量补列：CREATE TABLE IF NOT EXISTS 不会改动已存在的表，
 317|     # 因此新增列必须在这里对老库补齐（见 17.3 要点 4）
 318|     # 结构：{表名: {新增列名: 列定义}}，需求新增列时往这里加即可。
 319|     _NEW_COLUMNS: dict[str, dict[str, str]] = {"messages": {"intent": "TEXT"}}
 320| 
 321|     @classmethod
 322|     def _ensure_columns(cls, conn: sqlite3.Connection) -> None:
 323|         """为已存在的老表补上 SCHEMA 里新增、但老库没有的列。
 324| 
 325|         参数：
 326|             conn: 数据库连接（由 _write 传入）。
 327|         返回：
 328|             None。
 329|         说明：
 330|             PRAGMA table_info(表) 会列出该表当前的列，用列名集合判断缺哪列，
 331|             缺了才 ALTER TABLE ADD COLUMN（该操作在 SQLite 里安全、可重复执行）。
 332|         """
 333|         for table, columns in cls._NEW_COLUMNS.items():
 334|             existing = {row[1] for row in conn.execute(f"PRAGMA table_info({table})")}
 335|             for name, ddl in columns.items():
 336|                 if name not in

### 5.4 C 档：只点名，不展开

下面这些文件本节**只点名**，它们的主责在后面课次：

| 文件 | 职责 | 主责课次 |
|---|---|---|
| `core/infra/auth.py` | 签发 token、还原身份、密码哈希 | 第 38 节 |
| `core/infra/middleware.py` | `require_role` 依赖工厂：未登录 401、角色不符 403 | 第 38 节 |
| `core/api/auth_api.py` | 登录 / 登出 / 当前用户 | 第 38 节 |
| `core/api/chat_api.py` | 会话、消息、媒体上传与鉴权读取 | 第 38 节 |
| `core/api/agent_api.py` | 坐席工作台接口（会话 / 摘要 / 推荐） | 第 38 节 |
| `core/api/admin_api.py` | 管理后台（知识库 / 工单 / 日志 / 用户 / 评测） | 第 38 节 |
| `core/api/ws_api.py` | 三个 WebSocket 端点（user / agent / admin） | 第 38 节 |

> **为什么本节只点名**：它们属于"接口层"，第 38 节会把五组路由、
> 依赖注入与授权链、请求响应约定**系统地讲一遍**。本节只需确认"它们挂在哪一层"。

### 5.5 实验③：用 `ast` 把"模块地图"扫出来

架构图是**画的**，目录结构是**真的**。下面用 Python 标准库 `ast`
静态解析 `core/` 下每个 `.py`，把"目录 ↔ 层 ↔ 课次"自动对一遍。

In [10]:
# ============================================================
# Cell 10：实验③ —— 静态扫描 core/，生成"模块地图"
# ============================================================
import ast
from pathlib import Path

# 目录 → 层（__init__.py 不算教学模块，跳过）
LAYERS = [
    ("入口", [p.as_posix() for p in (Path("core/config.py"), Path("core/main.py"))]),
] + [
    (name, sorted(p.as_posix() for p in Path(f"core/{d}").glob("*.py")
                  if p.name != "__init__.py"))
    for name, d in [("路由层", "api"), ("基础设施层", "infra"), ("智能编排层", "ai"),
                    ("多模态 RAG 层", "rag"), ("业务编排层", "service")]
]

# 文件 → 主责课次（与 3.3 的表一致）
LESSON = {
    "config.py": "34", "main.py": "34", "db.py": "34", "init_data.py": "34", "scheduler.py": "34",
    "multimodal.py": "35", "media.py": "35", "llm.py": "35",
    "parsers.py": "36", "embedding.py": "36", "vector_store.py": "36",
    "multimodal_rag.py": "36", "seed_knowledge.py": "36",
    "crew_agents.py": "37", "crew_tools.py": "37", "handoff.py": "37",
    "session_flow.py": "37", "session_router.py": "37", "turn_context.py": "37",
    "mcp_client.py": "37", "mcp_bridge.py": "37", "async_bridge.py": "37",
    "auth.py": "38", "middleware.py": "38", "analytics.py": "38",
    "auth_api.py": "38", "chat_api.py": "38", "agent_api.py": "38",
    "admin_api.py": "38", "ws_api.py": "38", "ws_manager.py": "38",
}

total_files = total_lines = 0
for layer, files in LAYERS:
    print("=" * 80)
    print(f"  【{layer}】{len(files)} 个文件")
    print("=" * 80)
    print(f"  {'文件':<30}{'行':>5}{'类':>4}{'函数':>5}   主责   职责（docstring 首句）")
    print("  " + "-" * 76)
    for f in files:
        text = read(f)
        tree = ast.parse(text)
        n_class = sum(1 for n in tree.body if isinstance(n, ast.ClassDef))
        n_func = sum(1 for n in ast.walk(tree)
                     if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)))
        doc_lines = (ast.get_docstring(tree) or "").strip().splitlines()
        brief = doc_lines[0] if doc_lines else "(无 docstring)"
        if len(brief) > 28:
            brief = brief[:28] + "…"
        n_line = len(text.splitlines())
        total_files += 1
        total_lines += n_line
        print(f"  {f:<30}{n_line:>5}{n_class:>4}{n_func:>5}   "
              f"{LESSON.get(Path(f).name, '—'):<5}  {brief}")
    print()

print("=" * 80)
print(f"  合计：{total_files} 个模块文件，共 {total_lines} 行")
print("=" * 80)
print("  这就是第 34 节的「全项目地图」——后面每节课都只深入其中一格。")

  【入口】2 个文件
  文件                                行   类   函数   主责   职责（docstring 首句）
  ----------------------------------------------------------------------------
  core/config.py                  238   2   11   34     统一配置加载（能力四元组），见设计文档第十七章。
  core/main.py                    172   0    4   34     MDE-LAB 多模态数字员工平台 —— 中台服务入口。

  【路由层】5 个文件
  文件                                行   类   函数   主责   职责（docstring 首句）
  ----------------------------------------------------------------------------
  core/api/admin_api.py           813   6   31   38     管理端接口（见设计文档 12.5 / 11.3）。
  core/api/agent_api.py           251   3   12   38     坐席端接口（见设计文档 12.4）。
  core/api/auth_api.py            101   1    4   38     认证接口（见设计文档 11.4 / 11.6）。
  core/api/chat_api.py            226   2    9   38     用户端接口（见设计文档 12.3）。
  core/api/ws_api.py              285   0    7   38     WebSocket 接口（见设计文档 10.1 / 10…

  【基础设施层】10 个文件
  文件                                行   类   函数   主责   职责（docstring 首句）
  -------------------

---
## Part 6：把项目跑起来
---

### 6.1 启动检查清单

| 检查项 | 怎么看 |
|---|---|
| 依赖装好了 | `python -c "import fastapi, uvicorn, pydantic_settings"` |
| `.env` 就位 | 仓库根有 `.env`（不是只有 `.env.example`），且必填项不为空 |
| 必填四项 | `LLM_BASE_URL`、`LLM_MODEL`、`EMBEDDING_BASE_URL`、`EMBEDDING_MODEL` |
| 至少一个密钥 | `LLM_API_KEY` 或 `SILICONFLOW_API_KEY` |
| 目录可写 | `data/`（业务库 / 向量库 / 媒体）可创建 |

### 6.2 实验④a：让服务真正跑起来，再探一次活

在 notebook 里起服务有个坑：`uvicorn.run()` 会**阻塞内核**。
正确做法是**用子进程起服务**，主进程去探活，探完在 `finally` 里杀掉。

下面用 **8010 端口**（避开你本机可能已占用的 8000），探活 `/api/health`。
首次启动会做嵌入维度自检与 MCP 握手，可能要等几秒。

> ⚠️ 若你本机 8000 端口正在跑同一个项目也没关系：启动阶段不打开向量库
> （ChromaDB 是**用到才打开**），两个进程不会抢占向量库文件。

In [11]:
# ============================================================
# Cell 11：实验④a —— 子进程启动服务 + 探活 /api/health
# ============================================================
import os
import sys
import time
import json
import tempfile
import subprocess
from urllib.request import urlopen
from urllib.error import URLError

PORT = 8010
BASE = f"http://127.0.0.1:{PORT}"


def stop_process_tree(proc, timeout=10):
    """终止整棵进程树（含子进程）。

    为什么不能只用 proc.terminate()：uvicorn 启动时又会拉起「工单 MCP Server」子进程，
    而 terminate() 在 Windows 上只杀 uvicorn 自己 —— MCP 子进程会变成孤儿，
    继续占着我们重定向给它的日志文件句柄，于是后面的 os.unlink 会报 WinError 32。
    所以这里按「进程树」整棵终止。
    """
    if proc.poll() is None:
        if os.name == "nt":
            subprocess.run(["taskkill", "/F", "/T", "/PID", str(proc.pid)],
                           capture_output=True, check=False)
        else:
            proc.terminate()
    try:
        proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait(timeout=5)


def cleanup_log_file(fileobj):
    """关闭并尽量删除临时日志文件。

    Windows 上文件句柄可能被系统延迟释放，删不掉就留着（在系统临时目录，无副作用），
    绝不因为"清理失败"而让实验单元报错。
    """
    path = fileobj.name
    fileobj.close()
    try:
        os.unlink(path)
    except OSError:
        pass


env = dict(os.environ)                    # 继承当前环境
env["PYTHONPATH"] = os.getcwd()           # 子进程也能 import core.*
env.pop("MDE_SKIP_CONFIG_CHECK", None)    # 确保配置自检真的生效

log = tempfile.NamedTemporaryFile("w+", suffix=".log", delete=False, encoding="utf-8")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "core.main:app",
     "--host", "127.0.0.1", "--port", str(PORT), "--log-level", "info"],
    stdout=log, stderr=subprocess.STDOUT, cwd=os.getcwd(), env=env,
)

health = None
try:
    for i in range(60):                   # 最多等 60 秒（含嵌入自检与 MCP 握手）
        if proc.poll() is not None:
            print(f"[服务提前退出] 退出码 = {proc.returncode}")
            break
        try:
            with urlopen(f"{BASE}/api/health", timeout=2) as resp:
                health = json.loads(resp.read().decode("utf-8"))
                print(f"[就绪] 第 {i + 1} 次探活成功（约 {i + 1} 秒）")
                break
        except (URLError, ConnectionError, OSError):
            time.sleep(1)
finally:
    stop_process_tree(proc)               # 关键：杀整棵进程树，否则 MCP 子进程变孤儿

print()
print("=" * 62)
print("  GET /api/health 返回")
print("=" * 62)
if health:
    for k, v in health.items():
        print(f"  {k:<18} = {v}")
else:
    print("  （未取到健康检查结果，请看下面的服务日志）")

print()
print("=" * 62)
print("  服务日志（末尾 20 行）")
print("=" * 62)
with open(log.name, encoding="utf-8", errors="replace") as fh:   # with：读完立刻释放句柄
    log_text = fh.read()
for line in log_text.splitlines()[-20:]:
    print("  " + line)

cleanup_log_file(log)

[就绪] 第 3 次探活成功（约 3 秒）

  GET /api/health 返回
  status             = ok
  llm_configured     = True
  mcp_available      = True
  mcp_error          = None
  mcp_tools          = ['create_ticket', 'query_ticket', 'list_user_tickets', 'list_tickets', 'delete_ticket', 'update_ticket_status', 'reset_password', 'request_software_access']

  服务日志（末尾 20 行）
  └─────────────────────────────────────────────────────────────────────────────┘
  
  
  [10/07/26 10:42:20] INFO     Starting MCP server 'IT Service   transport.py:241
                               Desk' with transport 'stdio'                      
  2026-10-07 10:42:20,253 | INFO    | core.infra.mcp_client | 已连接工单 MCP Server：command=c:\Users\13691\Documents\Code\260601-GPKJ\07-MLM\.venv\Scripts\python.exe args=['mcp_servers/it_service/server.py'] cwd=C:\Users\13691\Documents\Code\260601-GPKJ\07-mlm
  2026-10-07 10:42:20,258 | INFO    | core.infra.mcp_bridge | 工单 MCP Server 已连接，工具：['create_ticket', 'query_ticket', 'list_user_tickets', 'li

### 6.3 实验④b：故意缺一项必填，观察"拒绝启动"

fail-fast 的价值在于**早失败、快修复**。下面用**环境变量**把 `LLM_MODEL` 抹空
（环境变量优先级高于 `.env`，所以**不会改动你的 `.env`**），再启动一次——
服务应当**拒绝启动**并明确报出缺哪个变量。

In [12]:
# ============================================================
# Cell 12：实验④b —— 缺必填项时，服务必须拒绝启动
# ============================================================
import os
import sys
import tempfile
import subprocess

env2 = dict(os.environ)
env2["PYTHONPATH"] = os.getcwd()
env2["LLM_MODEL"] = ""                    # 用环境变量覆盖 .env（环境变量优先级更高）
env2.pop("MDE_SKIP_CONFIG_CHECK", None)   # 不允许降级跳过自检

log2 = tempfile.NamedTemporaryFile("w+", suffix=".log", delete=False, encoding="utf-8")
proc2 = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "core.main:app",
     "--host", "127.0.0.1", "--port", "8011", "--log-level", "info"],
    stdout=log2, stderr=subprocess.STDOUT, cwd=os.getcwd(), env=env2,
)

try:
    code = proc2.wait(timeout=90)         # 期望它自己退出（拒绝启动）
    print(f"服务退出码 = {code}（非 0 即「拒绝启动」）")
except subprocess.TimeoutExpired:
    print("服务没有退出——说明自检没拦住（不符合预期，请检查 _REQUIRED）")
finally:
    stop_process_tree(proc2)              # 复用 Cell 11 的工具：没退出就强制收掉

with open(log2.name, encoding="utf-8", errors="replace") as fh:
    log_text2 = fh.read()
cleanup_log_file(log2)

print()
print("=" * 66)
print("  日志中的关键行（配置自检未通过 → 定位到 [LLM] LLM_MODEL）")
print("=" * 66)
for line in log_text2.splitlines():
    if any(k in line for k in ("配置自检未通过", "配置缺失", "[LLM]", "startup failed")):
        print("  " + line)

服务退出码 = 3（非 0 即「拒绝启动」）

  日志中的关键行（配置自检未通过 → 定位到 [LLM] LLM_MODEL）
  2026-10-07 10:42:24,496 | CRITICAL | mde-lab | 配置自检未通过，服务拒绝启动：
  配置缺失，请检查 .env：
    - [LLM] LLM_MODEL
          "配置缺失，请检查 .env：\n  - " + "\n  - ".join(missing)
  core.config.ConfigError: 配置缺失，请检查 .env：
    - [LLM] LLM_MODEL
  ERROR:    Application startup failed. Exiting.


---
## Part 7：随堂练习（含答案） + 小结
---

### 7.1 练习一：把 OCR 也变成"必填"

产品说："OCR 一定会配，缺了就别启动。"
请你在 `config.py` 的自检里加这一条，并验证它真的会报错。

> 提示：改动点在 `_REQUIRED` 列表；验证可以复用 Part 4 的 `simulate_missing()`。

In [13]:
# ============================================================
# Cell 13：练习一参考答案（可运行验证）
# ============================================================
import core.config as cfg

saved = list(cfg._REQUIRED)
try:
    # 只改一处：往必填清单里加一条（能力名, 变量名）
    cfg._REQUIRED = saved + [("OCR", "OCR_BASE_URL")]

    print("① 配齐 OCR_BASE_URL 时：")
    print("   " + simulate_missing(OCR_BASE_URL="http://example.com/v1").splitlines()[0])
    print()
    print("② 抹空 OCR_BASE_URL 时（应报出 [OCR] OCR_BASE_URL）：")
    print(simulate_missing(OCR_BASE_URL=""))
finally:
    cfg._REQUIRED = saved      # 还原，避免影响后续单元
    print()
    print("（已还原 _REQUIRED，本节其它单元不受影响）")

① 配齐 OCR_BASE_URL 时：
   通过（未报错）

② 抹空 OCR_BASE_URL 时（应报出 [OCR] OCR_BASE_URL）：
配置缺失，请检查 .env：
  - [OCR] OCR_BASE_URL

（已还原 _REQUIRED，本节其它单元不受影响）


### 7.2 练习二：为什么 ASR / OCR / VLM 不在必填清单里？

`_REQUIRED` 只有四项（LLM 与 EMBEDDING 各两项）。ASR、OCR、VLM 也是核心能力，
为什么不要求必填？

In [14]:
# ============================================================
# Cell 14：练习二参考答案
# ============================================================
print("""
结论：因为它们是「可降级能力」，而 LLM / EMBEDDING 是「主链路能力」。

1) LLM 与 EMBEDDING 是主链路：
   没有 LLM 就没法生成任何回复；没有 EMBEDDING 就没法建索引、没法检索。
   这两条断了，系统连"最小可用"都达不到 → 必须 fail-fast。

2) ASR / OCR / VLM 都有降级路径，缺了系统仍能跑：
   · ASR ：ASR_PROVIDER=api 不可用时可切 local（funasr）；员工也可以直接打字。
   · OCR ：OCR_PROVIDER=local 不可用时可复用 VLM；都没有则跳过图内文字提取。
   · VLM ：未配置时，知识库配图描述**自动跳过并回退页正文**，入库不中断
           （对应 KB_IMAGE_CAPTION_ENABLED 的兜底逻辑，见第 36 节）。

3) 这是「fail-fast 的边界」取舍：
   自检要拦的是"不修就跑不起来"的项；
   把可降级项也设成必填，只会让部署变脆——现场少配一个 VLM 就整个起不来。

一句话：必填清单 = 主链路的最小可用集，不是"功能全集"。
""")


结论：因为它们是「可降级能力」，而 LLM / EMBEDDING 是「主链路能力」。

1) LLM 与 EMBEDDING 是主链路：
   没有 LLM 就没法生成任何回复；没有 EMBEDDING 就没法建索引、没法检索。
   这两条断了，系统连"最小可用"都达不到 → 必须 fail-fast。

2) ASR / OCR / VLM 都有降级路径，缺了系统仍能跑：
   · ASR ：ASR_PROVIDER=api 不可用时可切 local（funasr）；员工也可以直接打字。
   · OCR ：OCR_PROVIDER=local 不可用时可复用 VLM；都没有则跳过图内文字提取。
   · VLM ：未配置时，知识库配图描述**自动跳过并回退页正文**，入库不中断
           （对应 KB_IMAGE_CAPTION_ENABLED 的兜底逻辑，见第 36 节）。

3) 这是「fail-fast 的边界」取舍：
   自检要拦的是"不修就跑不起来"的项；
   把可降级项也设成必填，只会让部署变脆——现场少配一个 VLM 就整个起不来。

一句话：必填清单 = 主链路的最小可用集，不是"功能全集"。



### 7.3 练习三：复述启动顺序与分层边界

不看代码，先自己回答，再对照答案：
1. `main.py` 的 lifespan 有哪五步？为什么配置自检必须排在第一步？
2. "路由层不写业务逻辑、不直接写 SQL"——这条边界带来什么好处？
3. `core/rag/` 与 `core/service/` 各管什么？为什么不合并？

In [ ]:
# ============================================================
# Cell 15：练习三参考答案
# ============================================================
print("""
1) lifespan 五步：
   ① 注册主事件循环（set_main_loop）
   ② 配置自检（fail-fast；可设 MDE_SKIP_CONFIG_CHECK=1 降级为告警）
   ③ 建库 + 差量补种演示账号
   ④ 预建工单库 + 拉起 MCP 常驻连接
   ⑤ 启动定时任务（+ 恢复中断的评测），最后挂路由、对外提供服务
   为什么自检排第一：配置是所有后续步骤的输入。配置错了，建库、连 MCP、起定时任务
   全是白做；而且晚失败更难定位（可能几十秒后才在某个请求里炸开）。

2) 路由层只做「收请求 → 鉴权 → 调服务 → 返回」：
   好处是业务逻辑只有一份，能被 HTTP、WebSocket、定时任务、Agent 工具
   多个入口共享调用；换入口不用重写逻辑，写测试也可以绕过 HTTP 直接测服务层。

3) 分工：
   · core/rag/     管「知识怎么进来、怎么被找到」（解析、嵌入、向量库、检索编排）
   · core/service/ 管「会话怎么流转」（意图 → 检索 → 生成 → 转人工 → 结束）
   不合并的原因：RAG 是「无状态的检索能力」，可被任何场景复用；
   业务编排是「有状态的流程」，依赖会话与用户。
   混在一起，就会出现"想换个知识库，却动到了会话状态机"。
""")


1) lifespan 五步：
   ① 注册主事件循环（set_main_loop）
   ② 配置自检（fail-fast；可设 MDE_SKIP_CONFIG_CHECK=1 降级为告警）
   ③ 建库 + 差量补种演示账号
   ④ 预建工单库 + 拉起 MCP 常驻连接
   ⑤ 启动定时任务（+ 恢复中断的评测），最后挂路由、对外提供服务
   为什么自检排第一：配置是所有后续步骤的输入。配置错了，建库、连 MCP、起定时任务
   全是白做；而且晚失败更难定位（可能几十秒后才在某个请求里炸开）。

2) 路由层只做「收请求 → 鉴权 → 调服务 → 返回」：
   好处是业务逻辑只有一份，能被 HTTP、WebSocket、定时任务、Agent 工具
   多个入口共享调用；换入口不用重写逻辑，写测试也可以绕过 HTTP 直接测服务层。

3) 分工：
   · core/rag/     管「知识怎么进来、怎么被找到」（解析、嵌入、向量库、检索编排）
   · core/service/ 管「会话怎么流转」（意图 → 检索 → 生成 → 转人工 → 结束）
   不合并的原因：RAG 是「无状态的检索能力」，可被任何场景复用；
   业务编排是「有状态的流程」，依赖会话与用户。
   混在一起，就会出现"想换个知识库，却动到了会话状态机"。



### 7.4 本节小结

| 主题 | 结论 | 落点 |
|---|---|---|
| 业务边界 | 三类输入（文字/语音/截图）× 两类出口（自助 / 转人工或建单） | — |
| 能力全景 | 理解类（ASR/OCR/VLM）+ 表达类（TTS）+ 嵌入/精排；**不做文生图** | `core/rag/multimodal.py`（第 35 节） |
| 五层架构 | `api / infra / ai / rag / service`，各层只做分内事 | `core/` |
| 配置治理 | 「能力四元组」+ Key 回退链，换供应商零改代码 | `core/config.py` |
| 启动自检 | 必填清单 fail-fast + 嵌入维度校验 | `validate_config()` |
| 启动编排 | lifespan 五步 + 应用装配 + SPA fallback | `core/main.py` |
| 存储 | 向量库（ChromaDB）/ 关系库（SQLite）/ 媒体目录 | `data/` |
| 外部系统 | 工单走 MCP：独立进程、常驻连接 | `mcp_servers/it_service/server.py` |

### 7.5 交付物 / 验收

- 能讲清业务边界与五层架构；
- `python -m core.main` 能启动，`/api/health` 返回 `status: ok`；
- **缺必填配置时启动失败**，且报错能定位到"哪个能力、哪个变量"；
- 能画出端到端数据流图，并说出每个环节归属哪节课。

### 7.6 下节课预告

**第 35 节：多模态核心能力开发与代码实现**
- ASR / TTS / OCR / VLM 四条链路怎么封装成工具层；
- 引擎抽象与 provider 选择，业务代码不判断操作系统；
- 媒体文件全生命周期：降采样 → 落盘 → 带鉴权访问。

---

## 📚 参考资料

- 课件：`第34节-多模态大模型基础与数字员工架构设计.pptx`
- 代码：`core/config.py` —— 能力四元组、Key 回退链、启动自检、嵌入维度校验
- 代码：`core/main.py` —— lifespan 启动编排、应用装配、SPA fallback
- 代码：`core/infra/db.py` —— SQLite 异步包装、写锁、差量补列
- 代码：`core/infra/init_data.py` —— 演示账号差量补种
- 代码：`core/infra/scheduler.py` —— 三类定时任务
- 文档：`docs/多模态数字员工平台.md` §3–§5、§17
- 文档：`docs/课程大纲.md` 第 34 节

> 🎉 恭喜你完成了项目启动课！
> 现在你不仅知道这个平台"长什么样"，更知道它**为什么这么搭、怎么才能跑起来**。